In [11]:
import json
import pandas as pd
import os

data = []
base_path = os.path.dirname(os.getcwd())
log_dir = os.path.join(base_path, "log")

for file in os.listdir(log_dir):
    if file.endswith(".json"):
        file_path = os.path.join(log_dir, file)
        with open(file_path) as f:
            for line in f:
                data.append(json.loads(line))

df = pd.DataFrame(data)

print(df.head())

                  eventid          src_ip  src_port         dst_ip  dst_port  \
0  cowrie.session.connect   101.126.41.73   58982.0  172.31.45.231    2222.0   
1   cowrie.client.version   101.126.41.73       NaN            NaN       NaN   
2       cowrie.client.kex   101.126.41.73       NaN            NaN       NaN   
3   cowrie.session.closed   101.126.41.73       NaN            NaN       NaN   
4  cowrie.session.connect  175.27.145.234   18854.0  172.31.45.231    2222.0   

        session protocol  \
0  8fb6f3d375c6      ssh   
1  8fb6f3d375c6      ssh   
2  8fb6f3d375c6      ssh   
3  8fb6f3d375c6      ssh   
4  3921794b8300      ssh   

                                                                             message  \
0   New connection: 101.126.41.73:58982 (172.31.45.231:2222) [session: 8fb6f3d375c6]   
1                                          Remote SSH version: SSH-2.0-libssh_0.11.1   
2                     SSH client hassh fingerprint: 03a80b21afa810682a776a7d42e5e6fb  

In [12]:
sessions = []
print(df.columns)
grouped = df.groupby("session") #Agrupas el df por sesión
for session_id, group in grouped:
    #Diccionario de esta sesion
    session_data = {}

    #Guarda ID de la sesión
    session_data["session"] = session_id

        # IP atacante
    session_data["src_ip"] = group["src_ip"].dropna().iloc[0] if not group["src_ip"].dropna().empty else None

    # Puerto origen 
    session_data["src_port"] = group["src_port"].dropna().iloc[0] if not group["src_port"].dropna().empty else None

        # Duración segun evento cierre
    duration = group[group["eventid"] == "cowrie.session.closed"]["duration"]
    session_data["duration"] = float(duration.iloc[0]) if not duration.empty else 0

    # Login exitoso (1 = si, 0 = no)
    session_data["login_success"] = int((group["eventid"] == "cowrie.login.success").any())

    # Intentos de login fallidos
    session_data["login_attempts"] = (group["eventid"] == "cowrie.login.failed").sum()

    # Nº comandos ejecutados por el atacante 
    session_data["num_commands"] = (group["eventid"] == "cowrie.command.input").sum()

    # Descarga de archivos en la sesión (1 = sí, 0 = no)
    session_data["file_download"] = int((group["eventid"] == "cowrie.session.file_download").any())

    #Comandos atacante
    commands = group[group["eventid"] == "cowrie.command.input"]["input"].dropna().tolist()
    session_data["commands"] = commands

    #Distintos ataques en base a comandos
    # 1. Reconocimiento
    session_data["recon"] = int(
        any(any(x in cmd.lower() for x in ["uname","whoami","id","ifconfig","pwd"]) for cmd in commands)
    )

    # 2. Descarga malware
    session_data["download"] = int(
        any(any(x in cmd.lower() for x in ["wget","curl","tftp"]) for cmd in commands)
    )

    # 3. Cambio permisos
    session_data["chmod"] = int(
        any("chmod" in cmd.lower() for cmd in commands)
    )

    # 4. Ejecución de binarios/scripts
    session_data["execution"] = int(
        any(any(x in cmd.lower() for x in ["./","sh ","bash "]) for cmd in commands)
    )

    # 5. Persistencia
    session_data["persistence"] = int(
        any(".ssh" in cmd.lower() or "authorized_keys" in cmd.lower() for cmd in commands)
    )

    #Variedad de comandos en el ataque
    session_data["unique_commands"] = len(set(commands)) 

    #Complejidad del comando
    if commands:
        avg_len = sum(map(len, commands)) / len(commands)
    else:
        avg_len = 0
    session_data["avg_command_length"] = avg_len

    # Velocidad del ataque, alto->bot, bajo -> humano
    session_data["commands_per_second"] = (
        session_data["num_commands"] / session_data["duration"]
        if session_data["duration"] > 0 else 0
    )

    # Agrega diccionario de sesión
    sessions.append(session_data)
    
# Crea dataframe a partir de todos los diccionarios
dataset = pd.DataFrame(sessions)

print(dataset.head(50))

Index(['eventid', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'session',
       'protocol', 'message', 'sensor', 'uuid', 'timestamp', 'version',
       'hassh', 'hasshAlgorithms', 'kexAlgs', 'keyAlgs', 'encCS', 'macCS',
       'compCS', 'langCS', 'duration', 'username', 'password', 'arch', 'input',
       'ttylog', 'size', 'shasum', 'duplicate', 'outfile', 'destfile', 'width',
       'height'],
      dtype='str')
         session           src_ip  src_port    duration  login_success  \
0   0033683edcd9      65.20.175.6   35567.0    7.600000              0   
1   00ae7ffbb20a    117.216.33.31   40544.0   11.400000              1   
2   010705ba79a9   35.130.111.146   35155.0  242.600000              1   
3   020c2cb3291e    120.194.50.39   12292.0    3.700000              0   
4   02a5f5f600de  213.130.207.177   51221.0    2.800000              0   
5   02c9c77c5411  120.234.232.184   10706.0   26.100000              0   
6   0400b1869c01     45.118.49.18   57987.0    3.100000          

In [13]:
# Cuenta de cuantas veces nos ha atacado cada IP.
group_by_ip = dataset["src_ip"].value_counts()
print(group_by_ip)


src_ip
101.126.41.73      35
45.194.37.246      28
102.210.149.130    21
103.236.140.19      7
199.58.185.11       6
183.247.171.186     5
94.156.14.80        5
162.81.54.105       4
186.215.204.109     4
120.194.50.39       3
101.66.165.103      3
113.108.88.121      3
47.84.130.19        3
64.72.74.162        3
49.124.153.36       3
192.111.143.93      3
183.171.56.193      3
179.185.29.233      3
113.160.140.138     3
103.59.94.117       3
220.93.167.144      3
180.163.62.126      3
82.65.140.218       3
179.185.18.67       3
65.20.175.6         2
213.130.207.177     2
120.234.232.184     2
8.218.235.126       2
68.7.114.69         2
106.89.66.114       2
180.94.74.82        2
220.246.42.217      2
67.201.59.67        2
192.252.210.237     2
183.82.108.109      2
183.233.85.194      2
218.94.115.164      2
182.60.128.241      2
179.184.218.49      2
36.92.35.211        2
47.92.240.93        2
197.155.225.93      2
113.46.192.66       2
95.165.142.8        2
72.37.216.65        2
74.

In [14]:
import geoip2.database

# Abrir la base de datos local
dic = {}
reader = geoip2.database.Reader('../db/GeoLite2-City.mmdb')
for i,k in  group_by_ip.items():

    response = reader.city(i)
    if response.country.name not in dic.keys():
        dic[response.country.name] = k
    else:
        dic[response.country.name] += k

reader.close()

print(dic)


{'China': 167, 'Seychelles': 28, 'South Africa': 21, 'Indonesia': 15, 'United States': 155, 'Bulgaria': 7, 'Brazil': 21, 'Singapore': 4, 'Malaysia': 40, 'Vietnam': 10, 'South Korea': 28, 'France': 7, 'Iraq': 29, 'Lithuania': 3, 'Hong Kong': 13, 'Afghanistan': 3, 'India': 60, 'Zimbabwe': 2, 'Russia': 31, 'Ethiopia': 7, 'Japan': 6, 'Pakistan': 2, 'Venezuela': 3, 'Taiwan': 12, 'United Kingdom': 5, 'Ireland': 3, 'Bangladesh': 5, 'Rwanda': 2, 'Colombia': 4, 'Iran': 4, 'New Zealand': 3, 'Ghana': 2, 'Sweden': 18, 'Canada': 4, 'Belarus': 1, 'Israel': 7, 'Türkiye': 5, 'Nicaragua': 1, 'Greece': 1, 'Iceland': 1, 'Thailand': 1, 'Norway': 1, 'Mexico': 2, 'Spain': 3, 'Germany': 4, 'Peru': 1, 'Italy': 1, 'Poland': 4, 'Hungary': 1, 'Kazakhstan': 2, 'Senegal': 1, 'United Arab Emirates': 1, 'Morocco': 1, 'Antigua and Barbuda': 1, 'Ukraine': 2, 'Australia': 1, 'Netherlands': 1, 'Mozambique': 1, 'Chile': 2, 'Tunisia': 1, 'Philippines': 1, 'Maldives': 1, 'Nigeria': 1}


In [15]:
# Comandos ejecutados

dataset_filtrado = dataset[dataset["commands"].str.len() > 0]
print(dataset_filtrado["commands"])

36                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [ ]:
low = 0
med = 0
high = 0
for i in dataset["duration"]:
    if i < 20:
        low += 1
    if 20 <= i < 80:
        med += 1
    if i >= 80:
        high += 1
print("Low duration connections: ", low)
print("Medium duration connections: ", med)
print("High duration connections: ", high)



Low duration connections:  631
Medium duration connections:  23
High duration connections:  121
